In [1]:
# ============================================================
# HOTEL BOOKING CANCELLATION PREDICTION AND DEPLOYMENT
# Streamlit Application
# ============================================================

# Install required packages once:
# pip install streamlit pandas numpy scikit-learn imbalanced-learn
# pip install xgboost joblib

# Run the application using:
# streamlit run app.py


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import os
import io
import joblib
import numpy as np
import pandas as pd
import streamlit as st


# ============================================================
# 2. PAGE CONFIGURATION
# ============================================================

st.set_page_config(
    page_title="Hotel Cancellation Predictor",
    page_icon="🏨",
    layout="wide"
)


# ============================================================
# 3. FILE PATHS
# ============================================================

MODEL_PATH = (
    "models/"
    "best_hotel_cancellation_model.pkl"
)

THRESHOLD_PATH = (
    "models/"
    "final_decision_threshold.pkl"
)

MODEL_INFORMATION_PATH = (
    "models/"
    "final_model_information.pkl"
)


# ============================================================
# 4. LOAD MODEL
# ============================================================

@st.cache_resource
def load_model():

    if not os.path.exists(MODEL_PATH):

        raise FileNotFoundError(
            f"Model file was not found at:\n{MODEL_PATH}"
        )

    loaded_model = joblib.load(
        MODEL_PATH
    )

    return loaded_model


# ============================================================
# 5. LOAD DECISION THRESHOLD
# ============================================================

@st.cache_resource
def load_threshold():

    if os.path.exists(THRESHOLD_PATH):

        threshold = joblib.load(
            THRESHOLD_PATH
        )

        return float(threshold)

    # Use the standard classification threshold
    # when a tuned threshold has not been saved.
    return 0.50


# ============================================================
# 6. LOAD MODEL INFORMATION
# ============================================================

@st.cache_resource
def load_model_information():

    if os.path.exists(
        MODEL_INFORMATION_PATH
    ):

        information = joblib.load(
            MODEL_INFORMATION_PATH
        )

        return information

    return {}


# ============================================================
# 7. LOAD DEPLOYMENT ASSETS
# ============================================================

try:

    model = load_model()
    default_threshold = load_threshold()
    model_information = load_model_information()

except Exception as error:

    st.error(
        "The deployment assets could not be loaded."
    )

    st.exception(error)
    st.stop()


# ============================================================
# 8. IDENTIFY EXPECTED MODEL FEATURES
# ============================================================

if hasattr(
    model,
    "feature_names_in_"
):

    expected_features = list(
        model.feature_names_in_
    )

else:

    st.error(
        "The saved pipeline does not contain "
        "feature_names_in_. Save the complete fitted pipeline, "
        "including the preprocessing stage."
    )

    st.stop()


# ============================================================
# 9. IDENTIFY CLASSIFIER
# ============================================================

if hasattr(
    model,
    "named_steps"
):

    if "classifier" in model.named_steps:

        classifier_name = type(
            model.named_steps[
                "classifier"
            ]
        ).__name__

    else:

        classifier_name = type(
            model
        ).__name__

else:

    classifier_name = type(
        model
    ).__name__


# ============================================================
# 10. DATA VALIDATION FUNCTION
# ============================================================

def validate_input_data(
    input_data,
    required_features
):
    """
    Check whether the uploaded dataset contains all
    features expected by the trained model.
    """

    data = input_data.copy()

    # Remove accidental index columns created by CSV files
    unnamed_columns = [
        column
        for column in data.columns
        if str(column).lower().startswith(
            "unnamed"
        )
    ]

    if unnamed_columns:

        data = data.drop(
            columns=unnamed_columns
        )

    missing_features = [
        feature
        for feature in required_features
        if feature not in data.columns
    ]

    extra_features = [
        feature
        for feature in data.columns
        if feature not in required_features
    ]

    if missing_features:

        return (
            False,
            data,
            missing_features,
            extra_features
        )

    # Select required columns and place them in
    # exactly the same order used during training.
    data = data[
        required_features
    ]

    return (
        True,
        data,
        missing_features,
        extra_features
    )


# ============================================================
# 11. FINAL PREDICTION FUNCTION
# ============================================================

def predict_cancellations(
    fitted_model,
    input_data,
    decision_threshold=0.50
):
    """
    Generate cancellation probabilities and labels.

    Prediction labels:
        0 = Not Cancelled
        1 = Cancelled
    """

    if not hasattr(
        fitted_model,
        "predict_proba"
    ):

        raise AttributeError(
            "The fitted model does not support predict_proba()."
        )

    probabilities = (
        fitted_model
        .predict_proba(
            input_data
        )[:, 1]
    )

    predictions = (
        probabilities
        >= decision_threshold
    ).astype(int)

    results = input_data.copy()

    results[
        "cancellation_probability"
    ] = probabilities

    results[
        "predicted_is_canceled"
    ] = predictions

    results[
        "prediction_label"
    ] = np.where(
        predictions == 1,
        "Likely to Cancel",
        "Unlikely to Cancel"
    )

    results[
        "prediction_confidence"
    ] = np.where(
        predictions == 1,
        probabilities,
        1 - probabilities
    )

    results[
        "risk_category"
    ] = pd.cut(
        probabilities,
        bins=[
            -0.01,
            0.25,
            0.50,
            0.75,
            1.00
        ],
        labels=[
            "Low Risk",
            "Moderate Risk",
            "High Risk",
            "Very High Risk"
        ]
    )

    results[
        "recommended_action"
    ] = np.select(
        [
            probabilities >= 0.75,

            (
                probabilities >= 0.50
            )
            &
            (
                probabilities < 0.75
            ),

            (
                probabilities >= 0.25
            )
            &
            (
                probabilities < 0.50
            )
        ],
        [
            (
                "Immediate confirmation, deposit review "
                "or targeted retention action"
            ),

            (
                "Contact the customer and confirm "
                "booking intention"
            ),

            (
                "Monitor the booking and send "
                "a confirmation reminder"
            )
        ],
        default=(
            "Standard booking management"
        )
    )

    return results


# ============================================================
# 12. CSV CONVERSION FUNCTION
# ============================================================

def convert_dataframe_to_csv(
    dataframe
):
    """
    Convert a dataframe into downloadable CSV bytes.
    """

    return dataframe.to_csv(
        index=False
    ).encode(
        "utf-8"
    )


# ============================================================
# 13. APPLICATION TITLE
# ============================================================

st.title(
    "🏨 Hotel Booking Cancellation Prediction"
)

st.write(
    """
    This application predicts whether a hotel booking is
    likely to be cancelled using a trained machine-learning
    pipeline.
    """
)


# ============================================================
# 14. SIDEBAR MODEL SETTINGS
# ============================================================

st.sidebar.header(
    "Model Settings"
)

selected_threshold = st.sidebar.slider(
    label="Decision threshold",
    min_value=0.05,
    max_value=0.95,
    value=float(
        round(
            default_threshold,
            2
        )
    ),
    step=0.01
)

st.sidebar.write(
    f"Loaded classifier: **{classifier_name}**"
)

st.sidebar.write(
    f"Number of expected features: "
    f"**{len(expected_features)}**"
)

st.sidebar.write(
    f"Default saved threshold: "
    f"**{default_threshold:.2f}**"
)


# ============================================================
# 15. MODEL INFORMATION
# ============================================================

with st.expander(
    "View model information"
):

    st.write(
        "Classifier:",
        classifier_name
    )

    st.write(
        "Pipeline type:",
        type(model).__name__
    )

    st.write(
        "Pipeline steps:",
        (
            list(
                model.named_steps.keys()
            )
            if hasattr(
                model,
                "named_steps"
            )
            else "Not available"
        )
    )

    if model_information:

        st.write(
            "Saved model information:"
        )

        st.json(
            model_information
        )


# ============================================================
# 16. REQUIRED FEATURE LIST
# ============================================================

with st.expander(
    "View required feature columns"
):

    feature_table = pd.DataFrame(
        {
            "Feature Number": range(
                1,
                len(expected_features) + 1
            ),

            "Required Feature":
                expected_features
        }
    )

    st.dataframe(
        feature_table,
        use_container_width=True,
        hide_index=True
    )


# ============================================================
# 17. DOWNLOAD INPUT TEMPLATE
# ============================================================

template_dataframe = pd.DataFrame(
    columns=expected_features
)

template_csv = convert_dataframe_to_csv(
    template_dataframe
)

st.download_button(
    label="Download prediction input template",
    data=template_csv,
    file_name=(
        "hotel_cancellation_"
        "prediction_template.csv"
    ),
    mime="text/csv"
)


# ============================================================
# 18. CREATE APPLICATION TABS
# ============================================================

batch_tab, single_tab = st.tabs(
    [
        "Batch Prediction",
        "Single Booking Prediction"
    ]
)


# ============================================================
# 19. BATCH PREDICTION TAB
# ============================================================

with batch_tab:

    st.subheader(
        "Upload Feature-Engineered Booking Data"
    )

    st.info(
        """
        Upload the feature-engineered dataset before one-hot
        encoding and scaling. The saved pipeline will perform
        imputation, encoding and scaling automatically.
        """
    )

    uploaded_file = st.file_uploader(
        label="Upload a CSV file",
        type=["csv"],
        key="batch_upload"
    )

    if uploaded_file is not None:

        try:

            uploaded_data = pd.read_csv(
                uploaded_file
            )

            st.success(
                "CSV file loaded successfully."
            )

            st.write(
                "Uploaded dataset shape:",
                uploaded_data.shape
            )

            st.write(
                "Uploaded data preview:"
            )

            st.dataframe(
                uploaded_data.head(20),
                use_container_width=True
            )

            (
                is_valid,
                prediction_data,
                missing_features,
                extra_features
            ) = validate_input_data(
                input_data=uploaded_data,
                required_features=expected_features
            )

            if extra_features:

                st.warning(
                    "The following extra columns were ignored:"
                )

                st.write(
                    extra_features
                )

            if not is_valid:

                st.error(
                    "The uploaded dataset is missing "
                    "required model features."
                )

                st.write(
                    "Missing features:"
                )

                st.write(
                    missing_features
                )

            else:

                st.success(
                    "All required model features are available."
                )

                if st.button(
                    "Generate Batch Predictions",
                    type="primary"
                ):

                    with st.spinner(
                        "Generating cancellation predictions..."
                    ):

                        prediction_results = (
                            predict_cancellations(
                                fitted_model=model,
                                input_data=
                                    prediction_data,
                                decision_threshold=
                                    selected_threshold
                            )
                        )

                    st.success(
                        "Predictions generated successfully."
                    )

                    # ----------------------------------------
                    # Prediction summary
                    # ----------------------------------------

                    total_bookings = len(
                        prediction_results
                    )

                    predicted_cancellations = int(
                        prediction_results[
                            "predicted_is_canceled"
                        ].sum()
                    )

                    predicted_non_cancellations = (
                        total_bookings
                        -
                        predicted_cancellations
                    )

                    average_probability = float(
                        prediction_results[
                            "cancellation_probability"
                        ].mean()
                    )

                    column_1, column_2, column_3, column_4 = (
                        st.columns(4)
                    )

                    column_1.metric(
                        "Total Bookings",
                        total_bookings
                    )

                    column_2.metric(
                        "Likely Cancellations",
                        predicted_cancellations
                    )

                    column_3.metric(
                        "Unlikely Cancellations",
                        predicted_non_cancellations
                    )

                    column_4.metric(
                        "Average Cancellation Risk",
                        f"{average_probability:.2%}"
                    )

                    # ----------------------------------------
                    # Display result columns first
                    # ----------------------------------------

                    result_columns = [
                        "predicted_is_canceled",
                        "prediction_label",
                        "cancellation_probability",
                        "prediction_confidence",
                        "risk_category",
                        "recommended_action"
                    ]

                    original_columns = [
                        column
                        for column in prediction_results.columns
                        if column not in result_columns
                    ]

                    display_results = (
                        prediction_results[
                            result_columns
                            +
                            original_columns
                        ]
                    )

                    st.subheader(
                        "Prediction Results"
                    )

                    st.dataframe(
                        display_results,
                        use_container_width=True
                    )

                    # ----------------------------------------
                    # Risk distribution
                    # ----------------------------------------

                    st.subheader(
                        "Cancellation Risk Distribution"
                    )

                    risk_distribution = (
                        prediction_results[
                            "risk_category"
                        ]
                        .value_counts()
                        .reindex(
                            [
                                "Low Risk",
                                "Moderate Risk",
                                "High Risk",
                                "Very High Risk"
                            ],
                            fill_value=0
                        )
                    )

                    st.bar_chart(
                        risk_distribution
                    )

                    # ----------------------------------------
                    # High-risk bookings
                    # ----------------------------------------

                    high_risk_bookings = (
                        prediction_results[
                            prediction_results[
                                "cancellation_probability"
                            ]
                            >= selected_threshold
                        ]
                        .sort_values(
                            "cancellation_probability",
                            ascending=False
                        )
                    )

                    st.subheader(
                        "Bookings Requiring Attention"
                    )

                    st.dataframe(
                        high_risk_bookings,
                        use_container_width=True
                    )

                    # ----------------------------------------
                    # Download predictions
                    # ----------------------------------------

                    prediction_csv = (
                        convert_dataframe_to_csv(
                            prediction_results
                        )
                    )

                    st.download_button(
                        label=(
                            "Download Prediction Results"
                        ),
                        data=prediction_csv,
                        file_name=(
                            "hotel_cancellation_"
                            "predictions.csv"
                        ),
                        mime="text/csv"
                    )

        except Exception as error:

            st.error(
                "An error occurred while processing "
                "the uploaded dataset."
            )

            st.exception(error)


# ============================================================
# 20. SINGLE BOOKING PREDICTION TAB
# ============================================================

with single_tab:

    st.subheader(
        "Predict One Booking"
    )

    st.write(
        """
        Upload a CSV containing exactly one feature-engineered
        booking. The file must contain all required input
        columns.
        """
    )

    single_booking_file = st.file_uploader(
        label="Upload one-row booking CSV",
        type=["csv"],
        key="single_upload"
    )

    if single_booking_file is not None:

        try:

            single_booking_data = pd.read_csv(
                single_booking_file
            )

            if len(single_booking_data) != 1:

                st.error(
                    "The single-booking CSV must contain "
                    "exactly one row."
                )

            else:

                (
                    single_valid,
                    single_prediction_data,
                    single_missing_features,
                    single_extra_features
                ) = validate_input_data(
                    input_data=
                        single_booking_data,

                    required_features=
                        expected_features
                )

                if single_extra_features:

                    st.warning(
                        "Extra columns were ignored:"
                    )

                    st.write(
                        single_extra_features
                    )

                if not single_valid:

                    st.error(
                        "The booking is missing required "
                        "model features."
                    )

                    st.write(
                        single_missing_features
                    )

                else:

                    st.write(
                        "Booking information:"
                    )

                    st.dataframe(
                        single_prediction_data,
                        use_container_width=True
                    )

                    if st.button(
                        "Predict This Booking",
                        type="primary"
                    ):

                        single_result = (
                            predict_cancellations(
                                fitted_model=model,
                                input_data=
                                    single_prediction_data,
                                decision_threshold=
                                    selected_threshold
                            )
                        )

                        probability = float(
                            single_result[
                                "cancellation_probability"
                            ].iloc[0]
                        )

                        prediction = int(
                            single_result[
                                "predicted_is_canceled"
                            ].iloc[0]
                        )

                        risk_category = str(
                            single_result[
                                "risk_category"
                            ].iloc[0]
                        )

                        recommended_action = str(
                            single_result[
                                "recommended_action"
                            ].iloc[0]
                        )

                        st.subheader(
                            "Final Prediction"
                        )

                        result_col_1, result_col_2, result_col_3 = (
                            st.columns(3)
                        )

                        result_col_1.metric(
                            "Cancellation Probability",
                            f"{probability:.2%}"
                        )

                        result_col_2.metric(
                            "Prediction",
                            (
                                "Likely to Cancel"
                                if prediction == 1
                                else "Unlikely to Cancel"
                            )
                        )

                        result_col_3.metric(
                            "Risk Category",
                            risk_category
                        )

                        if prediction == 1:

                            st.warning(
                                "This booking is predicted "
                                "to be at risk of cancellation."
                            )

                        else:

                            st.success(
                                "This booking is predicted "
                                "to remain active."
                            )

                        st.write(
                            "**Recommended action:**",
                            recommended_action
                        )

                        st.progress(
                            min(
                                max(
                                    probability,
                                    0.0
                                ),
                                1.0
                            )
                        )

                        st.dataframe(
                            single_result,
                            use_container_width=True
                        )

        except Exception as error:

            st.error(
                "The single booking could not be processed."
            )

            st.exception(error)


# ============================================================
# 21. APPLICATION FOOTER
# ============================================================

st.divider()

st.caption(
    """
    Predictions are decision-support estimates and should not
    be treated as guaranteed customer behaviour. Model
    performance should be monitored regularly for data drift,
    bias and changing booking patterns.
    """
)

2026-07-31 00:21:17.948 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-31 00:21:17.962 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-31 00:21:19.731 
  command:

    streamlit run C:\Users\ranpa\anaconda3\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-07-31 00:21:19.733 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-31 00:21:19.735 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-31 00:21:19.736 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-31 00:21:20.251 Thread 'Thread-3': missing ScriptRunContext! This warning can be ignored when runnin

DeltaGenerator()